# Messages
Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM. Messages are objects that contain:

Role - Identifies the message type (e.g. system, user)
Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
Metadata - Optional fields such as response information, message IDs, and token usage
LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In [3]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-120b")
model.invoke("What is the meaning of life?")

AIMessage(content='The question “What is the meaning of life?” is one of the oldest and most widely debated topics in human thought. Because it touches on philosophy, religion, science, and personal experience, there isn’t a single answer that satisfies everyone. Instead, people tend to approach it from a few broad perspectives. Below is a quick tour of the most common ways people think about life’s meaning, followed by some ideas for how you might explore the question for yourself.\n\n---\n\n## 1. Philosophical Views\n\n| School of Thought | Core Idea about Meaning |\n|-------------------|--------------------------|\n| **Existentialism** (e.g., Sartre, Camus) | Life has no pre‑given purpose; we create meaning through our choices and actions. |\n| **Absurdism** (Camus) | The universe is indifferent, but we can embrace the “absurd” and find personal joy despite it. |\n| **Nihilism** (e.g., Nietzsche’s early work) | There is no inherent meaning, and any search for it is futile. |\n| **Hu

### Text Prompts
Text prompts are strings - ideal for straightforward generation tasks where you don’t need to retain conversation history.

#### Use text prompts when:

You have a single, standalone request
You don’t need conversation history
You want minimal code complexity

In [4]:
model.invoke("What is the meaning of life?")

AIMessage(content='The question “What is the meaning of life?” has been asked for millennia, and there’s no single answer that satisfies everyone. Instead, people tend to approach it from a few broad angles—philosophical, religious, scientific, and personal. Below is a quick tour of some of the most common perspectives, followed by a few ideas for how you might explore the question for yourself.\n\n---\n\n## 1. Philosophical Views\n\n| School of Thought | Core Idea | Representative Thinkers |\n|-------------------|----------|--------------------------|\n| **Existentialism** | Life has no inherent meaning; we must create our own purpose through choices and authentic action. | Jean‑Paul Sartre, Albert Camus |\n| **Absurdism** | The universe is indifferent, and the search for meaning is inherently contradictory, yet we can find freedom in embracing the absurd. | Albert Camus |\n| **Utilitarianism** | Meaning comes from maximizing overall happiness or reducing suffering. | Jeremy Bentham, 

### Message Prompts
Alternatively, you can pass in a list of messages to the model by providing a list of message objects.

Message types

1. System message - Tells the model how to behave and provide context for interactions
2. Human message - Represents user input and interactions with the model
3. AI message - Responses generated by the model, including text content, tool calls, and metadata
4. Tool message - Represents the outputs of tool calls

### System Message
A SystemMessage represent an initial set of instructions that primes the model’s behavior. You can use a system message to set the tone, define the model’s role, and establish guidelines for responses.

### Human Message
A HumanMessage represents user input and interactions. They can contain text, images, audio, files, and any other amount of multimodal content.

### AI Message
An AIMessage represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.

### Tool Message
For models that support tool calling, AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.

In [6]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What is the meaning of life?"),
]
for chunk in model.stream(messages):
    print(chunk.content, end="")

The question “What is the meaning of life?” has been asked by thinkers, poets, scientists, and everyday people for millennia, and there’s no single answer that satisfies everyone. Instead, most perspectives fall into a few broad categories. Below is a quick tour of some of the most influential ways people have approached the question, along with a few ideas for how you might explore it for yourself.

---

## 1. Philosophical & Religious Traditions  

| Tradition | Core Idea about Meaning | Key Thinkers / Texts |
|-----------|------------------------|----------------------|
| **Western Philosophy (Ancient)** | Life’s purpose is found in virtue, reason, or the “good life.” | Socrates, Plato’s *Republic*, Aristotle’s *Nicomachean Ethics* (eudaimonia). |
| **Existentialism** | Meaning isn’t given; we must create it through authentic choices. | Jean‑Paul Sartre, Albert Camus (“The absurd”), Simone de Beauvoir. |
| **Stoicism** | Live in accordance with nature and reason; focus on what you c

In [9]:
system_message = SystemMessage("You are a Data scientist assistant.")
human_message = HumanMessage("What is Data Science? in Computer Science?")
response = model.invoke([system_message, human_message])
print(response.content)

**Data Science** is an interdisciplinary field that sits at the intersection of **computer science**, **statistics**, and **domain expertise**. In the context of computer science, it is the systematic study of how to extract knowledge and insights from data using computational methods. Below is a concise overview that frames data science within computer science.

---

## 1. Core Definition  

**Data Science** = *the practice of collecting, storing, processing, analyzing, visualizing, and interpreting large and complex data sets to support decision‑making, prediction, and automation.*

- **“Data”**: raw observations, logs, sensors, images, text, etc.  
- **“Science”**: a rigorous, reproducible methodology that includes hypothesis formulation, experimentation, validation, and communication of results.

---

## 2. Why It Belongs to Computer Science  

| Computer‑Science Aspect | Data‑Science Contribution |
|------------------------|---------------------------|
| **Algorithms & Data Struct

In [13]:
# Message metadata
human_message = HumanMessage(content="What is Data Science? in Computer Science?", name="mono", id="msg1")
res = model.invoke([human_message])
res.usage_metadata
# print(res.content)

{'input_tokens': 80,
 'output_tokens': 2792,
 'total_tokens': 2872,
 'output_token_details': {'reasoning': 67}}

In [14]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]

response = model.invoke(messages)
print(response.content)

2 + 2 = 4.


In [18]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)